# Functional Ensemble Detection — Fast Version (Directed Δ pR²)

**Speedups vs full notebook:**
- History = 200 ms (nfilters = 20 vs 100)
- 5-fold CV (vs 10)
- GC + NGS cells only (skips NS/Other)
- Upper triangle only — each pair computed once, **not** symmetrised

**Key difference from full notebook:**  
The Δ pR² matrix is kept **directed** (i→j ≠ j→i). Leiden runs on a directed graph.  
The directed asymmetry (i→j) − (j→i) is preserved and visualised separately.

**Caching:** inline pairwise results are saved to `data/eddie/` so re-running the  
notebook skips recomputation. Cache is bypassed when Eddie results are found.

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import glob, os, time, pickle
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from spatial_manifolds.detect_grids import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse        = 29
day          = 23
source_path  = '/Users/harryclark/Downloads/COHORT12/'
data_path    = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/xgboost_pairwise/'
cache_path   = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/leiden_fast_cache/'
fig_path     = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_playground/'
os.makedirs(cache_path, exist_ok=True)

# ── Speed parameters (fast notebook) ─────────────────────────────────────────
HISTORY_LENGTH  = 1000   # history length to look for in Eddie results
INLINE_HISTORY  = 200    # history used when computing inline
INLINE_CV_FOLDS = 3      # CV folds for inline computation
N_JOBS          = -1     # parallel workers (-1 = all cores)
CELL_TYPES      = ['GC', 'NG']   # restrict inline computation to these types

BASELINE_VR = 'null'
BASELINE_OF = 'null'

COL_GC    = '#c04744'
COL_NGS   = '#3171ae'
COL_OTHER = '#aaaaaa'

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
print(f'M{mouse} D{day}: {len(sess_cells)} classified cells')
print(f'VR baseline: {BASELINE_VR}  |  OF1 baseline: {BASELINE_OF}')
print(f'Inline history: {INLINE_HISTORY}ms  |  CV folds: {INLINE_CV_FOLDS}')

## 1. Load session data

In [ ]:
_t0 = time.time()
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False, source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')
print(f'  VR: {len(tcs_time_vr)} cells  {last_ephys_bin_vr * bs / 1000:.0f}s  ({time.time()-_t0:.1f}s)')

_t1 = time.time()
print('Loading OF1...')
tcs_of, tcs_time_of, beh_of, clusters_of, ep_of = compute_of_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path, session='OF1')
print(f'  OF1: {len(tcs_time_of)} cells  ({time.time()-_t1:.1f}s)')
print(f'Session data loaded in {time.time()-_t0:.1f}s total')

## 2. Load or compute pairwise Δ pR²

In [ ]:
import contextlib
import joblib
from tqdm.auto import tqdm

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Patch joblib batch callback to update a tqdm bar after each completed task."""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old
        tqdm_object.close()


def load_pairwise_results(mouse, day, history_length, session_type, data_path):
    pattern = os.path.join(
        data_path,
        f'xgboost_pairwise_M{mouse}_D{day}_h{history_length}_{session_type}_*.csv')
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'  Loaded {len(files)} file(s) → {len(df):,} rows  [h={history_length}]')
    return df


def extract_beh_arrays(beh, ep, T, session_type):
    """Pre-extract behavioral signals as plain numpy arrays (picklable)."""
    def _bin(key):
        arr = np.array(beh[key].bin_average(bin_size=time_bs, time_units='ms', ep=ep))
        if np.any(np.isnan(arr)):
            arr = pd.Series(arr).ffill().bfill().values
        return arr[:T]
    if session_type == 'VR':
        dt  = _bin('travel') - ((beh['trial_number'][0] - 1) * tl)
        return {'pos': (dt % tl), 'spd': _bin('S')}
    else:
        return {'px': _bin('P_x'), 'py': _bin('P_y'),
                'spd': _bin('S'), 'hd': _bin('H'), 'hing': _bin('Hing')}


def get_baseline_x(baseline, session_type, beh_arrays, T, lfp_trace=None):
    """Build behavioral covariate array from pre-extracted numpy dict."""
    if baseline == 'null':
        return np.zeros((T, 1))

    def _lfp():
        out = np.zeros(T)
        if lfp_trace is not None:
            n = min(len(lfp_trace), T); out[:n] = lfp_trace[:n]
        return out

    if session_type == 'VR':
        pos = beh_arrays['pos'];  spd = beh_arrays['spd'];  lfp = _lfp()
        mapping = {
            'pos': pos[:, None], 'speed': spd[:, None], 'lfp': lfp[:, None],
            'pos_speed': np.column_stack([pos, spd]),
            'pos_lfp': np.column_stack([pos, lfp]),
            'speed_lfp': np.column_stack([spd, lfp]),
            'pos_speed_lfp': np.column_stack([pos, spd, lfp]),
        }
    else:
        px = beh_arrays['px']; py = beh_arrays['py']; spd = beh_arrays['spd']
        hd = beh_arrays['hd']; hing = beh_arrays['hing']; lfp = _lfp()
        mapping = {
            'pos': np.column_stack([px, py]), 'speed': spd[:, None],
            'hd': hd[:, None], 'hing': hing[:, None], 'lfp': lfp[:, None],
            'pos_speed': np.column_stack([px, py, spd]),
            'pos_hd': np.column_stack([px, py, hd]),
            'pos_hing': np.column_stack([px, py, hing]),
            'pos_lfp': np.column_stack([px, py, lfp]),
            'pos_speed_hd': np.column_stack([px, py, spd, hd]),
            'pos_speed_hing': np.column_stack([px, py, spd, hing]),
            'pos_speed_lfp': np.column_stack([px, py, spd, lfp]),
            'pos_hd_hing': np.column_stack([px, py, hd, hing]),
            'pos_speed_hd_hing': np.column_stack([px, py, spd, hd, hing]),
            'pos_speed_hd_hing_lfp': np.column_stack([px, py, spd, hd, hing, lfp]),
        }
    if baseline not in mapping:
        raise ValueError(f'Unknown baseline "{baseline}" for {session_type}.')
    return mapping[baseline]


def _fit_one_target(ti, target_id, all_ids, spike_mat, T,
                    baseline, session_type, beh_arrays,
                    xgb, n_cv, mouse, day):
    """Fit baseline + all upper-triangle pairwise fits for one target cell."""
    rows = []

    # Per-cell LFP
    try:
        lfp_cell = np.array(get_theta_trace(
            mouse=mouse, day=day, cluster_id=target_id,
            time_bs=50, resample_bs=time_bs,
            session_type=session_type, source_path=source_path))
    except Exception as _e:
        lfp_cell = np.zeros(T)

    x_bl = get_baseline_x(baseline, session_type, beh_arrays, T, lfp_trace=lfp_cell)
    y    = spike_mat[target_id]

    # Baseline-only fit
    _, pr2_bl = xgb.fit_cv(x_bl, y, verbose=0, continuous_folds=True, n_cv=n_cv)
    rows.append(dict(
        mouse=mouse, day=day, session_type=session_type,
        target_cluster_id=int(target_id), target_type='unknown',
        covariate_cluster_id=-1, covariate_type='none',
        baseline=baseline, pR2_cv=float(np.nanmean(pr2_bl)),
    ))

    # Upper triangle: covariate index > target index
    for ci in range(ti + 1, len(all_ids)):
        cov_id = all_ids[ci]
        x = np.column_stack([x_bl, spike_mat[cov_id]])
        _, pr2_cv = xgb.fit_cv(x, y, verbose=0, continuous_folds=True, n_cv=n_cv)
        rows.append(dict(
            mouse=mouse, day=day, session_type=session_type,
            target_cluster_id=int(target_id), target_type='unknown',
            covariate_cluster_id=int(cov_id), covariate_type='unknown',
            baseline=baseline, pR2_cv=float(np.nanmean(pr2_cv)),
        ))
    print(f'  [worker] cell {target_id} done: {len(rows)} rows', flush=True)
    return rows


def compute_pairwise_inline(tcs_time, beh, ep, mouse, day, session_type,
                             baseline, history_length=INLINE_HISTORY,
                             n_cv=INLINE_CV_FOLDS, cell_types=CELL_TYPES,
                             n_jobs=N_JOBS, data_path=None):
    """
    Fast inline pairwise computation with joblib parallelism.
    Upper triangle only — directed matrix, do not symmetrise.
    Saves result to data_path so re-running loads from cache.
    """
    from joblib import Parallel, delayed
    from spatial_manifolds.mlencoding import MLencoding

    nfilters = int(history_length / time_bs)
    xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                     window=time_bs, n_filters=nfilters, max_time=history_length)

    all_ids = sorted(tcs_time.keys())

    # Filter to cell types of interest
    if cell_types is not None:
        try:
            gcs_s, ngs_s, all_s = classify_cells_both_sessions(
                mouse, day, source_path=source_path)
            keep = set()
            if 'GC' in cell_types: keep |= set(gcs_s.cluster_id.values.astype(int))
            if 'NG' in cell_types: keep |= set(ngs_s.cluster_id.values.astype(int))
            all_ids = [c for c in all_ids if c in keep]
            print(f'  Cell filter {cell_types}: {len(all_ids)} cells retained')
        except Exception as e:
            print(f'  Cell type filter failed ({e}), using all cells')

    T = min(len(np.array(tcs_time[cid])) for cid in all_ids)
    spike_mat = {cid: np.array(tcs_time[cid])[:T] for cid in all_ids}

    n_total = len(all_ids)
    n_pairs = n_total * (n_total - 1) // 2
    import multiprocessing
    n_cores = multiprocessing.cpu_count() if n_jobs == -1 else n_jobs
    print(f'  {n_total} cells  |  {n_pairs} pairs  |  {n_cores} workers')
    print(f'  history={history_length}ms  nfilters={nfilters}  cv={n_cv}-fold')
    print(f'  Est. at ~2s/fit / {n_cores} cores: {n_pairs * 2 / 3600 / n_cores:.1f} h')

    # Pre-extract beh signals as plain numpy arrays — h5py objects
    # cannot be pickled across joblib worker processes.
    beh_arrays = extract_beh_arrays(beh, ep, T, session_type)

    # ── Dry-run: test first cell synchronously before launching workers ──────
    print('  Dry-run: testing first cell synchronously...')
    _dry = _fit_one_target(0, all_ids[0], all_ids, spike_mat, T,
                           baseline, session_type, beh_arrays,
                           xgb, n_cv, mouse, day)
    print(f'  Dry-run OK: {len(_dry)} rows from cell {all_ids[0]}')

    _t_start = time.time()
    _desc = f'{session_type} [{baseline}] pairs'
    with tqdm_joblib(tqdm(desc=_desc, total=n_total,
                         unit='cell', dynamic_ncols=True)) as _bar:
        results = Parallel(n_jobs=n_jobs, prefer='processes', backend='loky')(
            delayed(_fit_one_target)(
                ti, target_id, all_ids, spike_mat, T,
                baseline, session_type, beh_arrays,
                xgb, n_cv, mouse, day)
            for ti, target_id in enumerate(all_ids)
        )

    rows = [row for target_rows in results for row in target_rows]
    print(f'  Done in {time.time()-_t_start:.1f}s')
    df = pd.DataFrame(rows)

    if data_path is not None:
        os.makedirs(data_path, exist_ok=True)
        out = os.path.join(
            data_path,
            f'xgboost_pairwise_M{mouse}_D{day}_h{history_length}_{session_type}_0_{n_total}.csv')
        df.to_csv(out, index=False)
        print(f'  Cached → {out}')
    return df


# ── Load or compute ────────────────────────────────────────────────────────────
_t_pw = time.time()
print(f'=== VR  [baseline: {BASELINE_VR}] ===')
pw_vr = load_pairwise_results(mouse, day, HISTORY_LENGTH, 'VR', data_path)
if pw_vr is None or BASELINE_VR not in pw_vr['baseline'].values:
    # No Eddie results — try inline cache (h=INLINE_HISTORY)
    pw_vr = load_pairwise_results(mouse, day, INLINE_HISTORY, 'VR', data_path)
    if pw_vr is None or BASELINE_VR not in pw_vr['baseline'].values:
        print('  No cache found — computing inline...')
        pw_vr = compute_pairwise_inline(
            tcs_time_vr, beh_vr, ep_vr, mouse, day, 'VR',
            baseline=BASELINE_VR, data_path=data_path)
    else:
        print('  Loaded from inline cache')

print(f'\n=== OF1 [baseline: {BASELINE_OF}] ===')
pw_of = load_pairwise_results(mouse, day, HISTORY_LENGTH, 'OF1', data_path)
if pw_of is None or BASELINE_OF not in pw_of['baseline'].values:
    pw_of = load_pairwise_results(mouse, day, INLINE_HISTORY, 'OF1', data_path)
    if pw_of is None or BASELINE_OF not in pw_of['baseline'].values:
        print('  No cache found — computing inline...')
        pw_of = compute_pairwise_inline(
            tcs_time_of, beh_of, ep_of, mouse, day, 'OF1',
            baseline=BASELINE_OF, data_path=data_path)
    else:
        print('  Loaded from inline cache')

print(f'\nVR  {len(pw_vr):,} rows  |  OF1 {len(pw_of):,} rows  ({time.time()-_t_pw:.1f}s)')

## 3. Build directed Δ pR² matrix

In [ ]:
def build_directed_matrix(pw_df, baseline):
    """
    Build an N×N DIRECTED Δ pR² matrix.
    mat[i, j] = pR²(baseline + cell_i) - pR²(baseline) predicting cell_j.
    i.e. row = covariate (predictor), column = target (predicted).

    When only the upper triangle was computed (fast mode), mat[j,i] stays NaN.
    This is intentional — do not symmetrise.
    """
    bl = (
        pw_df[(pw_df['baseline'] == baseline) & (pw_df['covariate_cluster_id'] == -1)]
        [['target_cluster_id', 'pR2_cv']]
        .rename(columns={'pR2_cv': 'pR2_baseline'})
        .drop_duplicates('target_cluster_id')
    )
    cv = (
        pw_df[(pw_df['baseline'] == baseline) & (pw_df['covariate_cluster_id'] >= 0)]
        [['target_cluster_id', 'covariate_cluster_id', 'pR2_cv']].copy()
    )
    merged = cv.merge(bl, on='target_cluster_id', how='inner')
    merged['delta_pr2'] = merged['pR2_cv'] - merged['pR2_baseline']

    all_ids   = sorted(set(merged['target_cluster_id']) | set(merged['covariate_cluster_id']))
    id_to_idx = {cid: i for i, cid in enumerate(all_ids)}
    N   = len(all_ids)
    mat = np.full((N, N), np.nan)

    for _, row in merged.iterrows():
        i = id_to_idx[int(row['covariate_cluster_id'])]  # predictor
        j = id_to_idx[int(row['target_cluster_id'])]     # predicted
        mat[i, j] = row['delta_pr2']

    np.fill_diagonal(mat, 0)
    return mat, all_ids


_t_mat = time.time()
print('Building directed matrices...')
dpr2_vr, ids_vr = build_directed_matrix(pw_vr, baseline=BASELINE_VR)
dpr2_of, ids_of = build_directed_matrix(pw_of, baseline=BASELINE_OF)
print(f'VR:  {dpr2_vr.shape}  NaN%={100*np.isnan(dpr2_vr).mean():.1f}%')
print(f'OF1: {dpr2_of.shape}  NaN%={100*np.isnan(dpr2_of).mean():.1f}%')
print(f'Built in {time.time()-_t_mat:.1f}s')
print()
# ~50% NaN expected when only upper triangle was computed
# (lower triangle entries are the mirror fits not yet computed)
print('Note: ~50% NaN expected — upper triangle only was computed.')
print('The available entries mat[i,j] tell us how well cell i predicts cell j.')

## 4. Directed Leiden (CPM)

In [ ]:
def run_leiden_directed(mat, n_search=300, res_min=0.5, res_max=1.75,
                        n_iter_search=10, n_iter_final=500, seed=42):
    """
    Leiden CPM on a DIRECTED weighted graph.
    NaN entries (uncomputed pairs) are treated as missing edges.
    Resolution search selects the parameter with highest modularity.
    """
    n = mat.shape[0]
    rows, cols = np.where(~np.isnan(mat) & (np.arange(n)[:, None] != np.arange(n)[None, :]))
    weights    = mat[rows, cols].tolist()

    g = ig.Graph(n=n,
                 edges=list(zip(rows.tolist(), cols.tolist())),
                 directed=True,
                 edge_attrs={'weight': weights})

    resolutions = np.linspace(res_min, res_max, n_search)
    best_mod, best_res = -np.inf, resolutions[0]
    mod_curve = []
    _t_res = time.time()
    print(f'  Searching {n_search} resolution values on directed graph ({n} nodes, {len(rows)} edges)...')

    for res in resolutions:
        part = leidenalg.find_partition(
            g, leidenalg.CPMVertexPartition,
            weights='weight', resolution_parameter=res,
            n_iterations=n_iter_search, seed=seed)
        mod_curve.append(part.modularity)
        if part.modularity > best_mod:
            best_mod, best_res = part.modularity, res

    print(f'  Search done in {time.time()-_t_res:.1f}s  |  best γ={best_res:.3f}  Q={best_mod:.3f}')
    _t_f = time.time()
    part = leidenalg.find_partition(
        g, leidenalg.CPMVertexPartition,
        weights='weight', resolution_parameter=best_res,
        n_iterations=n_iter_final, seed=seed)
    print(f'  Final run ({n_iter_final} iters) done in {time.time()-_t_f:.1f}s')

    labels = np.array(part.membership)
    unique, counts = np.unique(labels, return_counts=True)
    for u, c in zip(unique, counts):
        if c < 2:
            labels[labels == u] = -1

    return labels, best_res, best_mod, resolutions, np.array(mod_curve)


# ── Cache for Leiden labels ──────────────────────────────────────────────────
leiden_cache_file = os.path.join(
    cache_path, f'leiden_fast_M{mouse}D{day}_{BASELINE_VR}.pkl')

if os.path.exists(leiden_cache_file):
    print(f'Loading Leiden from cache: {leiden_cache_file}')
    with open(leiden_cache_file, 'rb') as _f:
        _cache = pickle.load(_f)
    labels_vr, res_vr, mod_vr, res_c_vr, mod_c_vr = _cache['vr']
    labels_of, res_of, mod_of, res_c_of, mod_c_of = _cache['of']
else:
    _t_leiden = time.time()
    print('Running directed Leiden (VR)...')
    labels_vr, res_vr, mod_vr, res_c_vr, mod_c_vr = run_leiden_directed(dpr2_vr)

    print('Running directed Leiden (OF1)...')
    labels_of, res_of, mod_of, res_c_of, mod_c_of = run_leiden_directed(dpr2_of)
    print(f'Leiden total: {time.time()-_t_leiden:.1f}s')

    with open(leiden_cache_file, 'wb') as _f:
        pickle.dump({'vr': (labels_vr, res_vr, mod_vr, res_c_vr, mod_c_vr),
                     'of': (labels_of, res_of, mod_of, res_c_of, mod_c_of)}, _f)
    print(f'Leiden cached → {leiden_cache_file}')

n_ens_vr = len(np.unique(labels_vr[labels_vr >= 0]))
n_ens_of = len(np.unique(labels_of[labels_of >= 0]))
print(f'\nVR:  {n_ens_vr} ensembles  (γ={res_vr:.3f}, Q={mod_vr:.3f})')
print(f'OF1: {n_ens_of} ensembles  (γ={res_of:.3f}, Q={mod_of:.3f})')

## 5. Ensemble membership

In [ ]:
ens_palette = plt.cm.tab20.colors
def ens_color(label): return ens_palette[label % len(ens_palette)] if label >= 0 else '#cccccc'

def make_member_df(ids, labels, sess_cells):
    cols = ['cluster_id', 'cell_type', 'probe_x', 'probe_y', 'SC_x', 'SC_y', 'SC_z', 'brain_region']
    avail = [c for c in cols if c in sess_cells.columns]
    df = pd.DataFrame({'cluster_id': ids, 'ensemble': labels})
    df = df.merge(sess_cells[avail], on='cluster_id', how='left')
    df['SC_x_abs'] = pd.to_numeric(df['SC_x'], errors='coerce').abs()
    df['medlat']   = np.where(df['SC_x_abs'] < 3400, 'medial', 'lateral')
    return df

df_vr = make_member_df(ids_vr, labels_vr, sess_cells)
df_of = make_member_df(ids_of, labels_of, sess_cells)

for session, df, n_ens in [('VR', df_vr, n_ens_vr), ('OF1', df_of, n_ens_of)]:
    print(f'\n── {session} ({n_ens} ensembles) ──')
    for ens in sorted(df[df['ensemble'] >= 0]['ensemble'].unique()):
        sub  = df[df['ensemble'] == ens]
        types = sub['cell_type'].value_counts().to_dict()
        ml    = sub['medlat'].value_counts().to_dict()
        print(f'  E{ens:2d}: n={len(sub):3d}  {types}  {ml}')

## 6. Figure

In [ ]:
def sort_by_ensemble(mat, labels):
    order = np.argsort(labels)
    return mat[np.ix_(order, order)], labels[order]

dpr2_vr_s, lbl_vr_s = sort_by_ensemble(dpr2_vr, labels_vr)
dpr2_of_s, lbl_of_s = sort_by_ensemble(dpr2_of, labels_of)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        hspace=0.42, wspace=0.38,
                        left=0.06, right=0.97, top=0.93, bottom=0.08)

for row_i, (session, mat_s, lbl_s, mat_dir, df, n_ens) in enumerate([
    ('VR',  dpr2_vr_s, lbl_vr_s, dpr2_vr, df_vr, n_ens_vr),
    ('OF1', dpr2_of_s, lbl_of_s, dpr2_of, df_of, n_ens_of),
]):
    # ── Sorted directed matrix ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[row_i, 0])
    vmax = np.nanpercentile(np.abs(mat_s), 98)
    im = ax.imshow(mat_s, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04).set_label('Δ pR² (i→j)', fontsize=7)
    for b in np.where(np.diff(lbl_s) != 0)[0] + 0.5:
        ax.axhline(b, color='black', lw=0.7); ax.axvline(b, color='black', lw=0.7)
    ax.set_title(f'Directed Δ pR² — {session}\n({n_ens} ensembles, upper tri only)',
                 fontsize=9, fontweight='bold', loc='left')
    ax.set_xlabel('Target (predicted)', fontsize=8)
    ax.set_ylabel('Covariate (predictor)', fontsize=8)
    ax.tick_params(labelsize=7)

    # ── Probe anatomy ─────────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[row_i, 1])
    ax.scatter(df[df['ensemble'] < 0]['probe_x'], df[df['ensemble'] < 0]['probe_y'],
               c='#cccccc', s=14, zorder=1)
    for ens in sorted(df[df['ensemble'] >= 0]['ensemble'].unique()):
        sub = df[df['ensemble'] == ens]
        ax.scatter(sub['probe_x'], sub['probe_y'], color=ens_color(ens),
                   s=24, zorder=2, edgecolors='k', linewidths=0.4, label=f'E{ens}')
    ax.legend(fontsize=6, frameon=False, ncol=2)
    ax.set_xlabel('Probe x (µm)', fontsize=8); ax.set_ylabel('Probe y (µm)', fontsize=8)
    ax.set_title(f'Probe anatomy ({session})', fontsize=9, fontweight='bold', loc='left')
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    # ── ML × DV scatter ───────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[row_i, 2])
    df_v = df[df['ensemble'] >= 0].copy()
    for ens in sorted(df_v['ensemble'].unique()):
        sub = df_v[df_v['ensemble'] == ens]
        ax.scatter(sub['SC_x_abs'], sub['SC_y'], color=ens_color(ens),
                   s=24, alpha=0.85, edgecolors='k', linewidths=0.3)
    ax.axvline(3400, color='black', lw=1.2, ls='--', alpha=0.7)
    ax.set_xlabel('|SC_x| ML (µm)', fontsize=8); ax.set_ylabel('SC_y DV (µm)', fontsize=8)
    ax.set_title(f'ML × DV anatomy ({session})', fontsize=9, fontweight='bold', loc='left')
    ax.spines[['top','right']].set_visible(False)
    ax.invert_yaxis(); ax.tick_params(labelsize=7)

plt.suptitle(
    f'Functional ensembles (fast) — M{mouse} D{day}  '
    f'Leiden directed CPM  h={INLINE_HISTORY}ms  baseline={BASELINE_VR}',
    fontsize=11, fontweight='bold')
plt.savefig(fig_path + f'leiden_fast_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 7. Save membership

In [ ]:
df_vr.to_csv(fig_path + f'leiden_fast_membership_VR_M{mouse}D{day}.csv',  index=False)
df_of.to_csv(fig_path + f'leiden_fast_membership_OF1_M{mouse}D{day}.csv', index=False)
print('Saved membership CSVs.')
df_vr[['cluster_id','ensemble','cell_type','medlat']].sort_values('ensemble')